<a href="https://colab.research.google.com/github/ehas1/Statistical-Bias-in-ML/blob/main/LIME_and_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Core Data Science Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Model Interpretability Libraries
import lime
import lime.lime_tabular
import shap

# ML Framework Libraries
import xgboost as xgb
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split

# Utility Libraries
import os
import glob
import joblib
from pathlib import Path
import requests
from io import StringIO

# Visualization Settings
sns.set_theme(style="whitegrid")
sns.set_context("paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Global Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# Storage and Model Loading Setup
def setup_storage():
    """Set up storage access for model files."""
    try:
        # Try to import and mount Google Drive
        from google.colab import drive
        print("Google Colab environment detected. Mounting Google Drive...")
        drive.mount('/content/drive')
        models_dir = "/content/drive/MyDrive/COMPAS_models"
    except ImportError:
        print("Local environment detected (not running in Colab)")
        models_dir = os.path.join(os.path.expanduser('~'), 'COMPAS_models')

    # Create the models directory if it doesn't exist
    if not os.path.exists(models_dir):
        os.makedirs(models_dir)
        print(f"Created models directory at {models_dir}")

    print(f"Using models directory: {models_dir}")
    return models_dir

def find_latest_model(pattern: str, models_dir: str = None) -> str:
    """Find the most recent model file matching the given pattern."""
    # Try multiple possible paths
    colab_path = "/content/drive/MyDrive/COMPAS_models"
    local_path = os.path.join(os.path.expanduser('~'), 'COMPAS_models')
    current_dir_path = os.path.join(os.getcwd(), 'COMPAS_models')

    # Use provided models_dir, or try all possible paths
    search_paths = [p for p in [models_dir, colab_path, local_path, current_dir_path] if p]

    print(f"\nSearching for pattern: {pattern}")
    print("Checking directories:")

    for path in search_paths:
        print(f"Checking {path}...")
        if os.path.exists(path):
            print(f"Directory exists: {path}")
            # List all files in directory
            print("Files in directory:")
            for f in os.listdir(path):
                print(f"  - {f}")

            # Try to find matching files
            full_pattern = os.path.join(path, pattern)
            files = glob.glob(full_pattern)
            if files:
                print(f"Found files matching {pattern}:")
                for f in files:
                    print(f"  - {f}")
                latest = max(files, key=os.path.getctime)
                print(f"Selected latest file: {latest}")
                # Check if it's a metadata file
                if 'metadata' in latest:
                    print("Warning: Selected file is a metadata file, skipping...")
                    continue
                return latest
        else:
            print(f"Directory does not exist: {path}")

    raise FileNotFoundError(f"No model files found matching pattern: {pattern}")

def load_latest_models() -> tuple:
    """Load the latest version of each model architecture."""
    try:
        # Find latest model files
        dt_model_path = find_latest_model('decision_tree_model_*.joblib')

        # Try to find XGBoost model with different possible extensions
        try:
            xgb_model_path = find_latest_model('xgboost_model_*[!metadata].ubj')
        except FileNotFoundError:
            try:
                xgb_model_path = find_latest_model('xgboost_model_*[!metadata].json')
            except FileNotFoundError:
                xgb_model_path = find_latest_model('xgboost_model_*[!metadata].*')

        nn_model_path = find_latest_model('neural_network_model_*.keras')

        print("Loading models from:")
        print(f"Decision Tree: {dt_model_path}")
        print(f"XGBoost: {xgb_model_path}")
        print(f"Neural Network: {nn_model_path}")

        # Load models using appropriate methods
        dt_model = joblib.load(dt_model_path)

        # Load XGBoost model
        try:
            wrapped_xgb = xgb.XGBClassifier(
                objective='binary:logistic',
                use_label_encoder=False
            )
            wrapped_xgb.load_model(str(xgb_model_path))
            print("Successfully loaded XGBoost model")
        except Exception as e:
            print(f"Error loading XGBoost model: {str(e)}")
            try:
                print("Attempting alternative loading method...")
                booster = xgb.Booster()
                booster.load_model(str(xgb_model_path))
                wrapped_xgb = xgb.XGBClassifier(
                    objective='binary:logistic',
                    use_label_encoder=False
                )
                wrapped_xgb._Booster = booster
                wrapped_xgb.n_classes_ = 2
                print("Successfully loaded XGBoost model using alternative method")
            except Exception as e2:
                print(f"Alternative loading method also failed: {str(e2)}")
                raise

        nn_model = load_model(nn_model_path)

        return dt_model, wrapped_xgb, nn_model

    except Exception as e:
        print(f"Error loading models: {str(e)}")
        print("Please ensure the models are saved in either:")
        print("- Google Colab: /content/drive/MyDrive/COMPAS_models/")
        print(f"- Local: {os.path.join(os.path.expanduser('~'), 'COMPAS_models')}")
        raise

# Set up storage and update the global models directory
MODELS_DIR = setup_storage()


In [ ]:
# Data Loading and Preprocessing Functions
def load_data(url="https://raw.githubusercontent.com/propublica/compas-analysis/refs/heads/master/cox-violent-parsed.csv"):
    """Load the COMPAS dataset from ProPublica's repository."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = pd.read_csv(StringIO(response.text))
        print(f"Successfully loaded data with shape: {data.shape}")
        return data
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        raise

def preprocess_data(data):
    """Preprocess the COMPAS dataset for model interpretation."""
    # Print available columns for debugging
    print("Available columns in dataset:", data.columns.tolist())

    # Define required columns and their expected categories
    required_features = [
        'age',
        'c_charge_degree',
        'race',
        'sex',
        'priors_count',
        'days_b_screening_arrest'
    ]
    target = 'is_recid'

    # Define expected categories for categorical variables
    expected_categories = {
        'c_charge_degree': ['F', 'M'],  # Felony and Misdemeanor
        'race': ['African-American', 'Caucasian', 'Hispanic', 'Other'],
        'sex': ['Female', 'Male']
    }

    # Validate required columns exist
    missing_cols = [col for col in required_features + [target] if col not in data.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Preprocessing pipeline
    print("\nPreprocessing steps:")

    # 1. Handle missing values
    print("1. Handling missing values...")
    for col in required_features:
        if data[col].isnull().any():
            if pd.api.types.is_numeric_dtype(data[col]):
                data[col].fillna(data[col].mean(), inplace=True)
            else:
                data[col].fillna(data[col].mode()[0], inplace=True)

    # 2. Convert categorical variables with controlled encoding
    print("2. Converting categorical variables...")
    data_processed = data[required_features].copy()

    # Ensure categories are mapped correctly
    data_processed['race'] = data_processed['race'].map(lambda x: 'Other' if x not in expected_categories['race'] else x)
    data_processed['c_charge_degree'] = data_processed['c_charge_degree'].map(lambda x: 'F' if x not in expected_categories['c_charge_degree'] else x)
    data_processed['sex'] = data_processed['sex'].map(lambda x: 'Male' if x not in expected_categories['sex'] else x)

    # Create dummy variables with specific categories
    categorical_features = []
    for cat_col, categories in expected_categories.items():
        # Create dummy variables with specific categories
        dummies = pd.get_dummies(
            data_processed[cat_col],
            prefix=cat_col,
            prefix_sep='_'
        )
        # Drop the first category to avoid multicollinearity
        dummies = dummies.drop(columns=[f"{cat_col}_{categories[0]}"])
        # Drop the original column and join the dummies
        data_processed = data_processed.drop(columns=[cat_col])
        data_processed = pd.concat([data_processed, dummies], axis=1)
        categorical_features.extend([col for col in dummies.columns])

    # 3. Normalize numerical features
    print("3. Normalizing numerical features...")
    numerical_features = ['age', 'priors_count', 'days_b_screening_arrest']
    for col in numerical_features:
        data_processed[col] = (data_processed[col] - data_processed[col].mean()) / data_processed[col].std()

    # 4. Split into features and target
    print("4. Preparing train-test split...")
    X = data_processed
    y = data[target]

    # Print feature columns for verification
    print("\nFeature columns after preprocessing:")
    print(X.columns.tolist())
    print(f"Number of features: {len(X.columns)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"\nPreprocessing complete:")
    print(f"Training set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")

    return (X_train, y_train), (X_test, y_test)

# Load and preprocess data
print("Loading and preprocessing data...")
data = load_data()
(X_train, y_train), (X_test, y_test) = preprocess_data(data)
feature_names = X_train.columns.tolist()


In [ ]:
# Load models and verify compatibility
def verify_model_compatibility(X_sample, models):
    """Verify that the preprocessed data matches the expected input shapes of the models."""
    print("Verifying model compatibility...")

    # Get the number of features in the preprocessed data
    n_features = X_sample.shape[1]
    print(f"\nPreprocessed data has {n_features} features")

    # Check Decision Tree model
    try:
        dt_pred = models['dt'].predict_proba(X_sample.values[:1])
        print("✓ Decision Tree model compatible")
    except Exception as e:
        print(f"✗ Decision Tree model error: {str(e)}")

    # Check XGBoost model
    try:
        xgb_pred = models['xgb'].predict_proba(X_sample.values[:1])
        print("✓ XGBoost model compatible")
    except Exception as e:
        print(f"✗ XGBoost model error: {str(e)}")

    # Check Neural Network model
    try:
        nn_input_shape = models['nn'].layers[0].input_shape[1]
        print(f"\nNeural Network expects {nn_input_shape} features")
        if nn_input_shape != n_features:
            print(f"✗ Neural Network input shape mismatch: expected {nn_input_shape}, got {n_features}")
            print("\nFeature columns in preprocessed data:")
            print(X_sample.columns.tolist())
            raise ValueError(f"Neural Network expects {nn_input_shape} features but got {n_features}")
        nn_pred = models['nn'].predict(X_sample.values[:1])
        print("✓ Neural Network model compatible")
    except Exception as e:
        print(f"✗ Neural Network model error: {str(e)}")

# Load models and verify compatibility
print("Loading models...")
dt_model, xgb_model, nn_model = load_latest_models()

models_dict = {
    'dt': dt_model,
    'xgb': xgb_model,
    'nn': nn_model
}

verify_model_compatibility(X_train, models_dict)


In [ ]:
# LIME and SHAP Setup Functions
def setup_explainers(X_train, feature_names, models):
    """Set up LIME and SHAP explainers for all models"""
    # LIME explainer
    lime_exp = lime.lime_tabular.LimeTabularExplainer(
        X_train.values,
        feature_names=feature_names,
        class_names=['No Recidivism', 'Recidivism'],
        mode='classification'
    )

    # SHAP explainers
    shap_exp = {
        'dt': shap.TreeExplainer(models['dt']),
        'xgb': shap.TreeExplainer(models['xgb']),
        'nn': shap.KernelExplainer(
            lambda x: models['nn'].predict(x.astype(np.float32)),
            shap.sample(X_train, 100, random_state=42),
            link="logit"
        )
    }

    return lime_exp, shap_exp

def plot_global_importance(X_train, lime_exp, shap_exp, models, feature_names):
    """Plot global feature importance using LIME and SHAP"""
    fig, axes = plt.subplots(2, 1, figsize=(15, 12))
    fig.suptitle('Global Feature Importance Comparison', fontsize=14)

    # SHAP importance
    plt.sca(axes[0])
    shap_values = {}
    for name, explainer in shap_exp.items():
        data = X_train if name != 'nn' else shap.sample(X_train, 100)
        values = explainer.shap_values(data)
        if isinstance(values, list):
            values = values[1]
        shap_values[name] = np.abs(values).mean(0)

    # Plot SHAP importance
    shap_df = pd.DataFrame(shap_values, index=feature_names)
    shap_df.plot(kind='bar', ax=axes[0])
    axes[0].set_title('SHAP Global Feature Importance')
    axes[0].set_xlabel('Features')
    axes[0].set_ylabel('Mean |SHAP value|')

    # LIME importance
    X_sample = X_train.sample(n=100, random_state=42)
    lime_values = {name: np.zeros(len(feature_names)) for name in models}

    for x in X_sample.values:
        for name, model in models.items():
            predict_fn = model.predict_proba if name != 'nn' else model.predict
            exp = lime_exp.explain_instance(x, predict_fn, num_features=len(feature_names))
            for feat, imp in exp.as_list():
                idx = feature_names.index(feat.split(' ')[0])
                lime_values[name][idx] += abs(imp)

    # Plot LIME importance
    lime_df = pd.DataFrame(lime_values, index=feature_names)
    lime_df.plot(kind='bar', ax=axes[1])
    axes[1].set_title('LIME Global Feature Importance')
    axes[1].set_xlabel('Features')
    axes[1].set_ylabel('Cumulative |importance|')

    plt.tight_layout()
    plt.show()

# Initialize explainers
print("Setting up LIME and SHAP explainers...")
lime_exp, shap_exp = setup_explainers(X_train, feature_names, models_dict)


In [ ]:
# Run LIME and SHAP Analysis
print("Generating global feature importance plots...")
plot_global_importance(X_train, lime_exp, shap_exp, models_dict, feature_names)

# Analyze random instances
print("\nAnalyzing 5 random instances...")
random_indices = np.random.choice(len(X_test), 5, replace=False)

for idx in random_indices:
    instance = X_test.iloc[idx]
    true_label = y_test.iloc[idx]

    print(f"\n{'='*80}")
    print(f"Instance {idx} (True label: {'Recidivism' if true_label == 1 else 'No Recidivism'})")
    print(f"{'='*80}")

    # Get predictions
    preds = {
        'dt': models_dict['dt'].predict_proba(instance.values.reshape(1, -1))[0, 1],
        'xgb': models_dict['xgb'].predict_proba(instance.values.reshape(1, -1))[0, 1],
        'nn': models_dict['nn'].predict(instance.values.reshape(1, -1))[0]
    }

    # Print predictions
    print("\nPredictions:")
    print("-" * 40)
    for name, pred in preds.items():
        print(f"{name.upper():15} : {pred:.3f}")

    # Create figure for visualizations
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle(f'Instance {idx} Analysis', fontsize=14)

    # Get LIME explanations
    for i, (name, model) in enumerate(models_dict.items()):
        predict_fn = model.predict_proba if name != 'nn' else model.predict
        exp = lime_exp.explain_instance(instance.values, predict_fn, num_features=6)

        # Plot LIME
        plt.sca(axes[0, i])
        exp.as_pyplot_figure(ax=axes[0, i])
        axes[0, i].set_title(f'LIME\n{name.upper()}')

        # Get and plot SHAP values
        shap_values = shap_exp[name].shap_values(instance.values.reshape(1, -1))
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

        expected_val = (shap_exp[name].expected_value[1]
                       if isinstance(shap_exp[name].expected_value, list)
                       else shap_exp[name].expected_value)

        plt.sca(axes[1, i])
        shap.force_plot(
            expected_val,
            shap_values[0],
            instance,
            feature_names=feature_names,
            matplotlib=True,
            show=False
        )
        axes[1, i].set_title(f'SHAP\n{name.upper()}')

    plt.tight_layout()
    plt.show()

    # Print feature importance details
    print("\nFeature Importance Details:")
    print("-" * 40)
    for name, model in models_dict.items():
        print(f"\n{name.upper()}:")

        # LIME explanation
        predict_fn = model.predict_proba if name != 'nn' else model.predict
        lime_exp_inst = lime_exp.explain_instance(instance.values, predict_fn, num_features=6)
        print("LIME Values:")
        for feat, imp in lime_exp_inst.as_list():
            print(f"  {feat:30} : {imp:+.3f}")

        # SHAP values
        shap_values = shap_exp[name].shap_values(instance.values.reshape(1, -1))
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        print("\nSHAP Values:")
        importances = list(zip(feature_names, shap_values[0]))
        for feat, imp in sorted(importances, key=lambda x: abs(x[1]), reverse=True)[:6]:
            print(f"  {feat:30} : {imp:+.3f}")
